# 02 — Analysis

Main effects in both directions, interactions against an explicit null, and
dispersion across seeds.

Two rules the tool enforces, worth remembering when reading the output: where
the from-below and from-above estimates of a main effect **disagree**, neither
may be quoted alone — the disagreement is the interaction. And nothing is
quotable without dispersion, so a partial campaign will show `nan` error bars
and `UNDETERMINED` rather than a confident-looking number.


## 1. Drive and repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ---------------------------------------------------------------------------
# The one place paths are defined. Everything else derives from DRIVE_ROOT.
#
#   e3dgsuw/
#     dataset/SeathruNeRF_dataset/   original, as downloaded
#     dataset/undistorted/<scene>/   PINHOLE + sparse/0/  <- required
#     dense/<scene>.ply|.json        M1 clouds, SHA-256 sidecars
#     run_ledger.json                campaign state
#     runs/<cell>/<scene>/s<seed>/   one run, all of it together
#     analysis/                      analyse.py output, figures, tables
# ---------------------------------------------------------------------------
DRIVE_ROOT   = '/content/drive/MyDrive/e3dgsuw'
DATA_ORIG    = f'{DRIVE_ROOT}/dataset/SeathruNeRF_dataset'
DATA_UNDIST  = f'{DRIVE_ROOT}/dataset/undistorted'
DENSE_DIR    = f'{DRIVE_ROOT}/dense'
ANALYSIS_DIR = f'{DRIVE_ROOT}/analysis'

# Training reads from local disk, not Drive: the scene loader pulls every image
# at startup, and Drive's FUSE layer makes that far slower than a single copy.
LOCAL_DATA   = '/content/data'

REPO_URL  = 'https://github.com/dinanirham/An-Efficient-3D-Gaussian-Splatting-for-Underwater-3D-Reconstruction.git'
REPO_DIR  = '/content/e3dgsuw'
IMPL_DIR  = f'{REPO_DIR}/implementation'
SCENES    = ['Curasao', 'IUI3-RedSea', 'JapaneseGradens-RedSea', 'Panama']

import os
for d in (DRIVE_ROOT, DATA_UNDIST, DENSE_DIR, ANALYSIS_DIR):
    os.makedirs(d, exist_ok=True)
print('drive root:', DRIVE_ROOT)


In [ ]:
import os, subprocess
if not os.path.exists(REPO_DIR):
    subprocess.run(['git','clone','--depth','1',REPO_URL,REPO_DIR], check=True)
else:
    subprocess.run(['git','-C',REPO_DIR,'pull','--ff-only'], check=True)
os.chdir(IMPL_DIR)
print(subprocess.run(['git','-C',REPO_DIR,'log','--oneline','-1'],
                     capture_output=True, text=True).stdout)

# Builds diff_gaussian_rasterization_ms and simple_knn for sm_80, and installs
# only the dependencies Colab does not already ship.
# no build needed: analysis is pure Python


## 2. Campaign state

In [ ]:
!python -m tools.run_ledger status --output_root "$DRIVE_ROOT"


## 3. Contrasts

Quality metrics combine additively; ratio measures (storage, primitive count,
frame rate) combine multiplicatively, in log space.

In [ ]:
!python -m tools.analyse \
    --output_root "$DRIVE_ROOT" \
    --weighting unweighted \
    --json "$ANALYSIS_DIR/analysis_unweighted.json"


In [ ]:
# Scene image counts are unequal (21/29/20/18), so the two weightings differ.
# Reporting both removes an easy source of disagreement.
!python -m tools.analyse \
    --output_root "$DRIVE_ROOT" \
    --weighting image_weighted \
    --json "$ANALYSIS_DIR/analysis_image_weighted.json"


## 4. The central hypothesis (H4)

Not a between-cell comparison: the depth normalisation constants and the medium
coefficients across the simplification boundary in A2. A large jump in β at
15 000 is the signature of the identifiability failure; its absorption inside
the re-identification burst is the signature of the fix.

In [ ]:
import glob, csv
import matplotlib.pyplot as plt

paths = sorted(glob.glob(f'{DRIVE_ROOT}/runs/A2/*/s0/diagnostics.csv'))
if not paths:
    print('No A2 runs yet — this is the S2 stage.')
for p in paths:
    rows = [r for r in csv.DictReader(open(p)) if r['beta_att_r']]
    if not rows:
        continue
    it  = [int(r['iteration']) for r in rows]
    br  = [float(r['beta_att_r']) for r in rows]
    zmx = [float(r['z_max']) if r['z_max'] else None for r in rows]
    scene = p.split('/runs/A2/')[1].split('/')[0]

    fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
    ax[0].plot(it, br); ax[0].axvline(15000, ls='--', c='r')
    ax[0].set_title(f'{scene}: beta_att (red)'); ax[0].set_xlabel('iteration')
    ax[1].plot(it, [z for z in zmx if z is not None][:len(it)])
    ax[1].axvline(15000, ls='--', c='r')
    ax[1].set_title('z_max (depth normalisation)'); ax[1].set_xlabel('iteration')
    plt.tight_layout(); plt.show()


## 5. Storage — per primitive as well as total

In [ ]:
# A ratio that fell because N fell would otherwise read as quantization
# performing worse. The codebook is a fixed cost, so the ratio grows with N.
!python -m tools.analyse --output_root "$DRIVE_ROOT" \
    --metric bytes_per_primitive --metric total_bytes --metric n_primitives_final


---
Outputs land in `analysis/` on Drive.
